# Pilot: throughput + tool-call error rate on Kaggle 2xT4
Research plan section 11.6 / 11.10 / 11.14. Measures episodes per session-hour and the tool-call parse-error rate for two open models served with llama.cpp, running AgentDojo's **no-attack utility** on `banking` and `slack`.

**Terms**: self-hosted Apache-2.0 weights; benchmark on fictional AgentDojo data; server stays on localhost; no tunnels (Kaggle AUP, section 11.2). Keep the notebook and any dataset **private**. One Kaggle account only.

Enable GPU **T4 x2** in the sidebar before running.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

## 1. Install pinned llama.cpp (CUDA 12.8) and agentdojo

In [ ]:
LLAMA_BUILD = 'b11344'
CUDA_TARBALL = 'https://github.com/ggml-org/llama.cpp/releases/download/b11344/llama-b11344-bin-ubuntu-cuda-12.8-x64.tar.gz'
CUDART_TARBALL = 'https://github.com/ggml-org/llama.cpp/releases/download/b11344/cudart-llama-b11344-bin-ubuntu-cuda-12.8-x64.tar.gz'
import os, subprocess
os.makedirs('/kaggle/working/llama', exist_ok=True)
for url in (CUDART_TARBALL, CUDA_TARBALL):
    fn = url.split('/')[-1]
    subprocess.run(['wget','-q','-O',f'/kaggle/working/llama/{fn}',url], check=True)
    subprocess.run(['tar','xzf',f'/kaggle/working/llama/{fn}','-C','/kaggle/working/llama'], check=True)
SERVER = '/kaggle/working/llama/build/bin/llama-server'
import glob
cands = glob.glob('/kaggle/working/llama/**/llama-server', recursive=True)
SERVER = cands[0]; print('llama-server at', SERVER)
print(subprocess.run([SERVER,'--version'],capture_output=True,text=True).stderr[:200])

In [ ]:
# pin agentdojo and deps
!pip -q install agentdojo==0.1.35 openai==3.23.0

## 2. Download the GGUFs (verified real repos, Apache-2.0)
* `ibm-granite/granite-4.2-8b-GGUF` -> `granite-4.2-8b-Q4_K_M.gguf` (~5.35 GB)
* `ggml-org/Qwen3.6-35B-A3B-GGUF` -> `Qwen3.6-35B-A3B-Q4_K_M.gguf` (~20.4 GB)

**Disk layout (2026-10-06 fix):** `/kaggle/working` is capped at ~20 GB and is reserved for `llama-server` binaries, run logs, and small outputs. Large GGUFs go under **scratch** `MODELS_DIR` = `/kaggle/tmp/models` if that exists, else `/tmp/models` (session-only). Do **not** download weights into `/kaggle/working/models` — a prior Run All filled working with Granite then died at ~67% of Qwen with ~13.7 GB free. Prefer Q4_K_M; only fall back to a smaller Qwen quant if scratch still cannot hold it (log any substitution in DECISIONS.md). If an exact file name has changed, list the repo and pick the nearest Q4_K_M.

In [ ]:
# Free /kaggle/working space left by a prior failed Qwen (or other) GGUF download.
import os, shutil, glob, subprocess
WORKING_MODELS = '/kaggle/working/models'
removed = []
if os.path.isdir(WORKING_MODELS):
    for pat in ['Qwen*', 'qwen*', '*.incomplete', '*.lock', '**/*Qwen*', '**/*.incomplete']:
        for fp in glob.glob(os.path.join(WORKING_MODELS, pat), recursive=True):
            try:
                if os.path.isdir(fp):
                    shutil.rmtree(fp); removed.append(fp + '/')
                elif os.path.isfile(fp) or os.path.islink(fp):
                    os.remove(fp); removed.append(fp)
            except FileNotFoundError:
                pass
    for root, dirs, files in os.walk(WORKING_MODELS, topdown=False):
        for name in files + dirs:
            if 'qwen' in name.lower() or name.endswith('.incomplete'):
                fp = os.path.join(root, name)
                try:
                    if os.path.isdir(fp): shutil.rmtree(fp)
                    else: os.remove(fp)
                    if fp not in removed: removed.append(fp)
                except FileNotFoundError:
                    pass
print('removed from /kaggle/working/models:', removed or '(nothing)')
print(subprocess.run(['df','-h','/kaggle/working','/tmp','/kaggle/tmp'],
                     capture_output=True, text=True).stdout)


In [ ]:
import os, hashlib, shutil, subprocess
from huggingface_hub import hf_hub_download

# Scratch for large weights (NOT /kaggle/working — ~20 GB cap caused prior ENOSPC)
for candidate in ('/kaggle/tmp/models', '/tmp/models'):
    parent = os.path.dirname(candidate)
    if os.path.isdir(parent) or parent == '/tmp':
        MODELS_DIR = candidate
        break
else:
    MODELS_DIR = '/tmp/models'
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['HF_HUB_CACHE'] = MODELS_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = MODELS_DIR

def free_gb(path):
    return shutil.disk_usage(path).free / (1024**3)

print(f'MODELS_DIR={MODELS_DIR}  free={free_gb(MODELS_DIR):.1f} GiB')
print(subprocess.run(['df','-h',MODELS_DIR,'/kaggle/working'],
                     capture_output=True, text=True).stdout)

QWEN_REPO = 'ggml-org/Qwen3.6-35B-A3B-GGUF'
QWEN_FILE = 'Qwen3.6-35B-A3B-Q4_K_M.gguf'  # prefer Q4_K_M (~20.4 GB)
if free_gb(MODELS_DIR) < 22.0:
    QWEN_FILE = 'Qwen3.6-35B-A3B-Q3_K_M.gguf'
    print(f'WARN: low scratch space; trying smaller quant {QWEN_FILE}')

GRANITE = hf_hub_download('ibm-granite/granite-4.2-8b-GGUF',
                          'granite-4.2-8b-Q4_K_M.gguf', local_dir=MODELS_DIR)
print('Granite ->', GRANITE)
QWEN = hf_hub_download(QWEN_REPO, QWEN_FILE, local_dir=MODELS_DIR)
print('Qwen ->', QWEN)

def sha256(p, n=1<<20):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        while (b:=f.read(n)): h.update(b)
    return h.hexdigest()
for p in (GRANITE, QWEN):
    print(p, f'{os.path.getsize(p)/(1024**3):.2f} GiB', sha256(p)[:16])

## 3. Serve a model and run no-attack utility
Granite 4.2-8B fits on one T4; Qwen3.6-35B-A3B (Q4_K_M ~20 GB) is split over both.

In [ ]:
import subprocess, time, os, requests, signal
def serve(model_path, ngl=99, ctx=24576, nslots=4, split=None, port=8099):
    env = dict(os.environ)
    cmd = [SERVER,'-m',model_path,'--host','127.0.0.1','--port',str(port),
           '-ngl',str(ngl),'-c',str(ctx),'-np',str(nslots),'--jinja','--temp','0']
    if split: cmd += ['--tensor-split', split]
    log = open(f'/kaggle/working/server_{port}.log','w')
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            if requests.get(f'http://127.0.0.1:{port}/v1/models').ok:
                print('server up'); return p
        except Exception: pass
        time.sleep(2)
    raise RuntimeError('server did not start; check the log')


In [ ]:
# attach this repo (add it as a Kaggle dataset or git clone) so attacks/scoring import
import sys; sys.path.insert(0, '/kaggle/working/code')  # adjust to where you put code/
import os; os.environ['LOCAL_LLM_PORT']='8099'
from agentdojo.task_suite.load_suites import get_suites
from agentdojo.agent_pipeline.agent_pipeline import AgentPipeline, PipelineConfig
from agentdojo.agent_pipeline.llms.openai_llm import OpenAILLM
from agentdojo.benchmark import benchmark_suite_without_injections
import openai, time


In [ ]:
def run_utility(model_path, label, split=None):
    from pathlib import Path
    from agentdojo.logging import OutputLogger
    print(f'{label}: loading weights / starting llama-server...', flush=True)
    proc = serve(model_path, split=split)
    try:
        client = openai.OpenAI(api_key='EMPTY', base_url='http://127.0.0.1:8099/v1')
        mid = client.models.list().data[0].id
        print(f'{label}: model id={mid}', flush=True)
        llm = OpenAILLM(client, mid)
        pipe = AgentPipeline.from_config(PipelineConfig(llm=llm, model_id=mid,
               defense=None, system_message_name=None, system_message=None))
        pipe.name = label
        suites = get_suites('v1.2.2')
        results = {}
        t0 = time.time(); n_eps = 0
        # Official layout: runs/{pipeline_name}/suite/... (D19). Do not nest label in logdir.
        logdir = Path('/kaggle/working/runs')
        logdir.mkdir(parents=True, exist_ok=True)
        with OutputLogger(str(logdir)):
            for s in ['banking','slack']:
                print(f'{label}: starting {s} utility (no attack)...', flush=True)
                t_suite = time.time()
                r = benchmark_suite_without_injections(pipe, suites[s],
                        logdir=logdir, force_rerun=False,
                        benchmark_version='v1.2.2')
                u = list(r['utility_results'].values())
                n_eps += len(u)
                results[s] = sum(u)/len(u) if u else float('nan')
                print(f'{label}/{s}: utility={results[s]:.3f}, n={len(u)}, {time.time()-t_suite:.0f}s', flush=True)
        dt = time.time()-t0
        eph = n_eps/(dt/3600) if dt > 0 else 0.0
        print(f'{label}: utility {results}, episodes {n_eps}, {dt:.0f}s, {eph:.1f} episodes/h', flush=True)
        model_dir = logdir / label
        n_json = nbytes = 0
        if model_dir.is_dir():
            for fp in model_dir.rglob('*.json'):
                n_json += 1
                try: nbytes += fp.stat().st_size
                except OSError: pass
        print(f'{label}: traces under {model_dir}: {n_json} json files, {nbytes/1024:.1f} KiB', flush=True)
        return results, eph
    finally:
        proc.terminate()
        print(f'{label}: server stopped', flush=True)


In [ ]:
# Granite (1 GPU) then Qwen (split across 2 GPUs)
run_utility(GRANITE, 'granite-4.2-8b')
run_utility(QWEN, 'qwen3.6-35b-a3b', split='1,1')

## 4. Tool-call parse-error rate
Strict check aligned with AgentDojo EmptyFunctionCall / tool-call parse failure markers (research-plan §11.8: replace a model if parse errors >5%). Does **not** flag the bare substring `"error"` (that false-positive flagged 37/37 in pilot Version #1 — D18). Scans JSON under `/kaggle/working/runs/{label}/`.

In [ ]:
import json, glob, re
from pathlib import Path

# Precise markers only. Do NOT match bare 'error' (D18).
PARSE_MARKERS = (
    'EmptyFunctionCall',
    'could not parse function call',
    'Invalid function call',
    'Invalid function calling output',
    'ASTParsingError',
    'Error parsing the model output',
    '<empty-function-name>',
    'could not parse',
)
_MARKER_RE = re.compile('|'.join(re.escape(m) for m in PARSE_MARKERS), re.IGNORECASE)

def _episode_has_parse_error(d):
    top = d.get('error')
    if isinstance(top, str) and top and _MARKER_RE.search(top):
        return True
    for msg in d.get('messages') or []:
        if not isinstance(msg, dict):
            continue
        err = msg.get('error')
        if isinstance(err, str) and err and _MARKER_RE.search(err):
            return True
        content = msg.get('content')
        chunks = []
        if isinstance(content, str):
            chunks.append(content)
        elif isinstance(content, list):
            for b in content:
                if isinstance(b, dict) and isinstance(b.get('content'), str):
                    chunks.append(b['content'])
                elif isinstance(b, str):
                    chunks.append(b)
        for c in chunks:
            if _MARKER_RE.search(c):
                return True
    return False

def parse_error_rate(label, threshold=0.05):
    files = sorted(glob.glob(f'/kaggle/working/runs/{label}/**/*.json', recursive=True))
    tot = err = 0
    flagged = []
    for fp in files:
        try:
            d = json.load(open(fp))
        except Exception:
            continue
        if not isinstance(d, dict) or 'messages' not in d:
            continue
        tot += 1
        if _episode_has_parse_error(d):
            err += 1
            flagged.append(fp)
    rate = err / tot if tot else float('nan')
    over = (rate > threshold) if tot else False
    print(f"{label}: parse-errors {err}/{tot} = {rate:.1%}  "
          f"({'EXCEEDS' if over else 'within'} §11.8 {threshold:.0%} replacement rule)")
    if flagged[:5]:
        print(f'  examples: {flagged[:5]}')
    return rate

for lab in ['granite-4.2-8b', 'qwen3.6-35b-a3b']:
    parse_error_rate(lab)


## 5. Trace inventory (keep JSON in Kaggle Output)
List `/kaggle/working/runs` so Save & Run All commits per-episode AgentDojo JSON (D19).

In [ ]:
import json, os
from pathlib import Path

runs = Path('/kaggle/working/runs')
manifest = {'runs_root': str(runs), 'models': {}}
print(f'runs root exists={runs.is_dir()}')
if runs.is_dir():
    for model_dir in sorted(p for p in runs.iterdir() if p.is_dir()):
        files = list(model_dir.rglob('*.json'))
        nbytes = sum(f.stat().st_size for f in files if f.is_file())
        manifest['models'][model_dir.name] = {
            'n_json': len(files), 'bytes': nbytes,
            'sample': [str(f.relative_to(runs)) for f in files[:5]],
        }
        print(f'  {model_dir.name}: {len(files)} json, {nbytes/1024:.1f} KiB')
        for f in files[:3]:
            print(f'    e.g. {f.relative_to(runs)}')
else:
    print('  WARNING: /kaggle/working/runs missing — traces will not appear in Output')

man_path = Path('/kaggle/working/runs_manifest.json')
man_path.write_text(json.dumps(manifest, indent=2))
print(f'wrote {man_path} ({man_path.stat().st_size} bytes)')
print('/kaggle/working top-level:')
for p in sorted(Path('/kaggle/working').iterdir()):
    if p.name.startswith('.'): continue
    print(f"  {'dir' if p.is_dir() else 'file'} {p.name}")


### Record in DECISIONS.md
Measured episodes/session-hour per model, no-attack utility on banking+slack, **strict** tool-call parse-error rate (err/tot and whether >5%), and the Gemma go/no-go draft. Re-run `scoop/cost_noapproval.py` with the measured rates (section 11.6). See D17–D19.